# Load the Product Catalog

In [16]:
# Product Recommendation - Load Product Catalog

import pandas as pd
import numpy as np

print("Libraries loaded successfully.")

product_path = "../cleaned_data/product_catalog_cleaned.csv"

products = pd.read_csv(product_path)

print("Product catalog loaded successfully.")

print("\nShape:")
print(products.shape)

print("\nColumns:")
print(products.columns.tolist())

print("\nFirst 5 rows:")
display(products.head())

Libraries loaded successfully.
Product catalog loaded successfully.

Shape:
(838, 18)

Columns:
['product_id', 'product_name', 'category', 'sub_category', 'is_fragile', 'is_hazmat', 'requires_cold_chain', 'stock_qty', 'avg_rating', 'tags', 'launch_date', 'specs.weight_kg', 'specs.battery_included', 'price.currency', 'price.amount', 'specs.dimensions_cm.length', 'specs.dimensions_cm.width', 'specs.dimensions_cm.height']

First 5 rows:


,product_id,product_name,category,sub_category,is_fragile,is_hazmat,requires_cold_chain,stock_qty,avg_rating,tags,launch_date,specs.weight_kg,specs.battery_included,price.currency,price.amount,specs.dimensions_cm.length,specs.dimensions_cm.width,specs.dimensions_cm.height
0,PRD-00453,Zenith Saree 2024,Apparel,Saree,False,False,False,230,3.38,['bestseller'],2022-07-04 00:00:00+00:00,0.74,Unknown,INR,93078.54,106.95,58.7,59.0
1,PRD-00517,Nexon Running Shoes Pro,Apparel,Running Shoes,False,False,False,407,3.30,"['fragile-handle', 'clearance']",2020-11-27 00:00:00+00:00,0.89,False,INR,133860.89,102.90,115.1,3.5
2,PRD-00524,Meridian Saree Max,Apparel,Saree,False,False,False,536,2.71,"['eco', 'new', 'fragile-handle']",2022-06-17 00:00:00+00:00,0.79,Unknown,inr,25610.55,145.00,50.6,25.1
3,PRD-00190,Kaveri Microwave,Home Appliance,Microwave,False,False,False,331,4.37,"['premium', 'new']",2022-03-15 00:00:00+00:00,43.13,True,inr,163381.89,17.20,87.9,9.9
4,PRD-00107,Meridian Smartphone Max,Electronics,Smartphone,False,False,False,596,2.83,[],2023-10-07 00:00:00+00:00,1.28,Unknown,INR,157038.82,36.70,65.8,123.6


# Prepare recommendation features

In [20]:
# PRODUCT RECOMMENDATION - PREPARE PRODUCT FEATURES

recommendation_data = products[
    [
        "product_id",
        "product_name",
        "category",
        "sub_category",
        "tags",
        "avg_rating",
        "specs.weight_kg",
        "is_fragile",
        "is_hazmat",
        "requires_cold_chain",
        "specs.battery_included"
    ]
].copy()

# Convert tags to text
recommendation_data["tags"] = (
    recommendation_data["tags"]
    .astype(str)
)

# Create a combined feature text
recommendation_data["feature_text"] = (
    recommendation_data["category"].astype(str) + " " +
    recommendation_data["sub_category"].astype(str) + " " +
    recommendation_data["tags"]
)

print("Recommendation features prepared successfully.")

display(
    recommendation_data[
        [
            "product_id",
            "product_name",
            "category",
            "sub_category",
            "feature_text"
        ]
    ].head()
)

Recommendation features prepared successfully.


,product_id,product_name,category,sub_category,feature_text
0,PRD-00453,Zenith Saree 2024,Apparel,Saree,Apparel Saree ['bestseller']
1,PRD-00517,Nexon Running Shoes Pro,Apparel,Running Shoes,"Apparel Running Shoes ['fragile-handle', 'clea..."
2,PRD-00524,Meridian Saree Max,Apparel,Saree,"Apparel Saree ['eco', 'new', 'fragile-handle']"
3,PRD-00190,Kaveri Microwave,Home Appliance,Microwave,"Home Appliance Microwave ['premium', 'new']"
4,PRD-00107,Meridian Smartphone Max,Electronics,Smartphone,Electronics Smartphone []


# Create TF-IDF features

In [21]:
# PRODUCT RECOMMENDATION - TF-IDF FEATURE MATRIX

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(
    stop_words="english"
)

tfidf_matrix = tfidf.fit_transform(
    recommendation_data["feature_text"]
)

print("TF-IDF model created successfully.")
print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF model created successfully.
TF-IDF matrix shape: (838, 62)


# Create Cosine Similarity Matrix

In [22]:
# PRODUCT RECOMMENDATION - CALCULATE SIMILARITY

similarity_matrix = cosine_similarity(tfidf_matrix)

print("Cosine similarity matrix created successfully.")
print("Similarity matrix shape:", similarity_matrix.shape)

Cosine similarity matrix created successfully.
Similarity matrix shape: (838, 838)


# Recommendation Function

In [24]:
# PRODUCT RECOMMENDATION - FINAL RECOMMENDATION FUNCTION

def recommend_products(product_id, top_n=5, min_rating=0, in_stock_only=False):

    if product_id not in recommendation_data["product_id"].values:
        return f"Product ID {product_id} not found."

    product_index = recommendation_data[
        recommendation_data["product_id"] == product_id
    ].index[0]

    similarity_scores = list(
        enumerate(similarity_matrix[product_index])
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    # Remove the selected product
    similarity_scores = [
        item for item in similarity_scores
        if item[0] != product_index
    ]

    # Create recommendation dataframe
    recommendations = recommendation_data.iloc[
        [item[0] for item in similarity_scores]
    ].copy()

    recommendations["similarity_score"] = [
        item[1] for item in similarity_scores
    ]

    # Stock information
    recommendations = recommendations.merge(
        products[["product_id", "stock_qty"]],
        on="product_id",
        how="left"
    )

    # Rating filter
    recommendations = recommendations[
        recommendations["avg_rating"] >= min_rating
    ]

    # Stock filter
    if in_stock_only:
        recommendations = recommendations[
            recommendations["stock_qty"] > 0
        ]

    return recommendations[
        [
            "product_id",
            "product_name",
            "category",
            "sub_category",
            "avg_rating",
            "stock_qty",
            "similarity_score"
        ]
    ].head(top_n).reset_index(drop=True)


print("Final recommendation function created successfully.")

Final recommendation function created successfully.


# Test the Recommendation System

In [25]:
# PRODUCT RECOMMENDATION - FINAL TEST

test_product_id = "PRD-00453"

print(f"Top 5 recommendations for {test_product_id}")
print("Minimum rating: 4.0")
print("In-stock products only\n")

result = recommend_products(
    product_id=test_product_id,
    top_n=5,
    min_rating=4.0,
    in_stock_only=True
)

display(result)

print("\nNumber of recommendations:", len(result))
print("Duplicate product IDs:", result["product_id"].duplicated().sum())

Top 5 recommendations for PRD-00453
Minimum rating: 4.0
In-stock products only



,product_id,product_name,category,sub_category,avg_rating,stock_qty,similarity_score
0,PRD-00455,Tamira Saree,Apparel,Saree,4.34,622,0.922161
1,PRD-00432,Nexon Saree Classic,Apparel,Saree,4.47,394,0.911860
2,PRD-00499,Meridian Saree Classic,Apparel,Saree,4.11,725,0.868009
3,PRD-00506,Tamira Saree Lite,Apparel,Saree,4.94,1065,0.828415
4,PRD-00514,Meridian Saree Classic,Apparel,Saree,4.89,178,0.770407



Number of recommendations: 5
Duplicate product IDs: 0


# Product Comparison

In [10]:
# PRODUCT RECOMMENDATION - PRODUCT COMPARISON

def compare_products(product_ids):
    
    available_ids = set(products["product_id"])
    
    invalid_ids = [
        product_id
        for product_id in product_ids
        if product_id not in available_ids
    ]
    
    if invalid_ids:
        return f"Product ID(s) not found: {invalid_ids}"
    
    comparison_columns = [
        "product_id",
        "product_name",
        "category",
        "sub_category",
        "avg_rating",
        "stock_qty",
        "specs.weight_kg",
        "specs.battery_included",
        "is_fragile",
        "is_hazmat",
        "requires_cold_chain",
        "price.currency",
        "price.amount"
    ]
    
    comparison = products[
        products["product_id"].isin(product_ids)
    ][comparison_columns].copy()
    
    return comparison.reset_index(drop=True)


print("Product comparison function created successfully.")

Product comparison function created successfully.


# Test Product Comparison

In [11]:
# PRODUCT RECOMMENDATION - TEST PRODUCT COMPARISON

product_1 = "PRD-00453"
product_2 = "PRD-00455"

print(f"Comparing {product_1} and {product_2}:")

display(
    compare_products(
        [product_1, product_2]
    )
)

Comparing PRD-00453 and PRD-00455:


,product_id,product_name,category,sub_category,avg_rating,stock_qty,specs.weight_kg,specs.battery_included,is_fragile,is_hazmat,requires_cold_chain,price.currency,price.amount
0,PRD-00453,Zenith Saree 2024,Apparel,Saree,3.38,230,0.74,Unknown,False,False,False,INR,93078.54
1,PRD-00455,Tamira Saree,Apparel,Saree,4.34,622,1.12,False,True,False,False,INR,111503.47


# Error Handling Test

In [26]:
# PRODUCT RECOMMENDATION - INVALID PRODUCT TEST

invalid_product_id = "PRD-99999"

result = recommend_products(
    product_id=invalid_product_id,
    top_n=5
)

print("Invalid Product Test Result:")
print(result)

Invalid Product Test Result:
Product ID PRD-99999 not found.
